# 01 — Data Cleaning & Quality Checks
**Project:** Customer Profitability & Revenue Leakage Intelligence

In [3]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()

if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent

RAW_FILE = ROOT / "data" / "raw" / "Sample - Superstore.csv"
OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("CSV path:", RAW_FILE)
print("CSV exists:", RAW_FILE.exists())


Project root: A:\Data Analyst\Customer-profitability-intellegence
CSV path: A:\Data Analyst\Customer-profitability-intellegence\data\raw\Sample - Superstore.csv
CSV exists: True


In [4]:
if not RAW_FILE.exists():
    raise FileNotFoundError(f"CSV not found: {RAW_FILE}. Run from the project root or update RAW_FILE.")

for enc in ("utf-8-sig", "cp1252", "latin1"):
    try:
        df = pd.read_csv(RAW_FILE, encoding=enc)
        print("Loaded with encoding:", enc)
        break
    except UnicodeDecodeError:
        continue
else:
    raise ValueError("Could not decode the CSV.")

df.columns = [str(c).strip() for c in df.columns]
print(f"Rows: {len(df):,}; columns: {len(df.columns)}")
display(df.head())


Loaded with encoding: cp1252
Rows: 9,994; columns: 21


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11-08-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11-08-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,06-12-2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10-11-2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10-11-2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [5]:
print("Missing values by column:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("Exact duplicate rows:", int(df.duplicated().sum()))

Missing values by column:


,missing_count
Row ID,0
Order ID,0
Order Date,0
Ship Date,0
Ship Mode,0
Customer ID,0
Customer Name,0
Segment,0
Country,0
City,0


Exact duplicate rows: 0


In [6]:
for col in ["Order Date", "Ship Date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

for col in ["Sales", "Profit", "Quantity", "Discount"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["Order ID", "Customer ID", "Product ID"]:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

display(df.dtypes.astype(str).to_frame("dtype"))


,dtype
Row ID,int64
Order ID,string
Order Date,datetime64[ns]
Ship Date,datetime64[ns]
Ship Mode,object
Customer ID,string
Customer Name,object
Segment,object
Country,object
City,object


In [7]:
for col in ["Sales", "Profit", "Quantity", "Discount"]:
    if col in df.columns:
        s = df[col]
        print(f"{col}: missing/non-numeric={s.isna().sum():,}, min={s.min()}, max={s.max()}")
if "Sales" in df.columns:
    print("Negative sales rows:", int((df["Sales"] < 0).sum()))
if "Profit" in df.columns:
    print("Loss-making rows:", int((df["Profit"] < 0).sum()))
if "Quantity" in df.columns:
    print("Non-positive quantity rows:", int((df["Quantity"] <= 0).sum()))
if "Discount" in df.columns:
    print("Discount outside 0–1:", int(((df["Discount"] < 0) | (df["Discount"] > 1)).sum()))


Sales: missing/non-numeric=0, min=0.444, max=22638.48
Profit: missing/non-numeric=0, min=-6599.978, max=8399.976
Quantity: missing/non-numeric=0, min=1, max=14
Discount: missing/non-numeric=0, min=0.0, max=0.8
Negative sales rows: 0
Loss-making rows: 1871
Non-positive quantity rows: 0
Discount outside 0–1: 0


In [8]:
out = OUT_DIR / "superstore_cleaned.csv"
df.to_csv(out, index=False)
print("Saved cleaned copy:", out)
print("Rows retained:", f"{len(df):,}")


Saved cleaned copy: A:\Data Analyst\Customer-profitability-intellegence\data\processed\superstore_cleaned.csv
Rows retained: 9,994
